# Santorini V2 Supervised Bootstrap on Kaggle

Use this notebook with a Kaggle GPU accelerator to train the V2 anonymous-worker architecture from the migrated Iteration 250 checkpoint and translated replay buffer.

Expected input Dataset contents:

- `best.pth.tar`: migrated V2 checkpoint with transferred residual/value weights and a fresh 1600-action policy head.
- `merged_20.examples`: translated V2 replay buffer with 20 history windows.

The local source artifacts are currently:

- `temp/santorini_kaggle_training6_v2/best.pth.tar`
- `temp/santorini_kaggle_training6_v2/merged_20.examples`

## 0. P100 PyTorch Compatibility Guard

Run this before importing `torch`. Kaggle's preinstalled CUDA 12.8 PyTorch wheel may not support Tesla P100 (`sm_60`), so this cell switches only P100 sessions to the official CUDA 12.6 wheel.

In [ ]:
import subprocess
import sys

try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
    ).strip()
except Exception:
    gpu_name = ""

print("GPU:", gpu_name or "not detected")

if "P100" in gpu_name:
    print("Detected P100; installing CUDA 12.6 PyTorch wheel...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "torch",
        "torchvision",
        "torchaudio",
    ])
else:
    print("No P100-specific PyTorch reinstall needed.")

## 1. Check Runtime

In [ ]:
import os
import torch

print("/kaggle/working exists:", os.path.exists("/kaggle/working"))
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Get the Repo

Enable Internet in the Kaggle notebook settings before running the clone and pip cells. If Internet is disabled, attach the repo as a Dataset instead and set `REPO_DIR` to that copied or mounted path.

In [ ]:
REPO_URL = "https://github.com/Luminous9/alpha-zero-general.git"
REPO_DIR = "/kaggle/working/alpha-zero-general"
# Set this to your V2 branch name if these changes are not on the default branch yet.
REPO_BRANCH = "main"

In [ ]:
%cd /kaggle/working
!test -d "$REPO_DIR/.git" && git -C "$REPO_DIR" fetch --all || git clone "$REPO_URL" "$REPO_DIR"
!git -C "$REPO_DIR" checkout "$REPO_BRANCH"
!git -C "$REPO_DIR" pull --ff-only || true
%cd $REPO_DIR
!git rev-parse --short HEAD

## 3. Install Dependencies

In [ ]:
!pip install coloredlogs tqdm

## 4. Configure Bootstrap Inputs

Attach the Kaggle Dataset that contains the V2 migration artifacts, then set `BOOTSTRAP_SOURCE` to that read-only `/kaggle/input/...` folder. The notebook creates symlinks in `/kaggle/working` so the 2.6 GB replay file does not need to be copied.

In [ ]:
import os
import shutil

# Example: "/kaggle/input/santorini-v2-bootstrap"
BOOTSTRAP_SOURCE = "/kaggle/input/santorini-v2-bootstrap"
BOOTSTRAP_WORK = "/kaggle/working/Santorini-AZ/v2_bootstrap"
BOOTSTRAP_CHECKPOINT = "best.pth.tar"
BOOTSTRAP_EXAMPLES = "merged_20.examples"

os.makedirs(BOOTSTRAP_WORK, exist_ok=True)

def link_or_copy(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    try:
        os.symlink(src, dst)
        print("Symlinked", dst, "->", src)
    except OSError:
        shutil.copy2(src, dst)
        print("Copied", src, "->", dst)

for filename in [BOOTSTRAP_CHECKPOINT, BOOTSTRAP_EXAMPLES]:
    src = os.path.join(BOOTSTRAP_SOURCE, filename)
    dst = os.path.join(BOOTSTRAP_WORK, filename)
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    link_or_copy(src, dst)

!ls -lh "$BOOTSTRAP_WORK"

## 5. Validate Inputs

In [ ]:
import pickle
import numpy as np

examples_path = os.path.join(BOOTSTRAP_WORK, BOOTSTRAP_EXAMPLES)
with open(examples_path, "rb") as f:
    history = pickle.load(f)

print("history windows:", len(history))
print("window lengths:", [len(window) for window in history])
print("total examples:", sum(len(window) for window in history))
print("first policy shape:", history[0][0][1].shape)
print("first policy sum:", float(np.sum(history[0][0][1])))

## 6. Supervised Bootstrap

This trains from the migrated V2 checkpoint using the translated replay buffer. Start with 3 epochs. If memory is tight, lower `BOOTSTRAP_BATCH_SIZE` to 128 or 64.

In [ ]:
BOOTSTRAP_EPOCHS = 3
BOOTSTRAP_BATCH_SIZE = 256
BOOTSTRAPPED_FILE = "bootstrapped.pth.tar"

In [ ]:
!python migrate_santorini_v2.py \
  --source-folder "$BOOTSTRAP_WORK" \
  --output-folder "$BOOTSTRAP_WORK" \
  --skip-checkpoint \
  --skip-examples \
  --output-checkpoint-file "$BOOTSTRAP_CHECKPOINT" \
  --output-examples-file "$BOOTSTRAP_EXAMPLES" \
  --bootstrap-epochs "$BOOTSTRAP_EPOCHS" \
  --bootstrap-batch-size "$BOOTSTRAP_BATCH_SIZE" \
  --bootstrapped-checkpoint-file "$BOOTSTRAPPED_FILE"

## 7. Sanity Check Bootstrapped Checkpoint

In [ ]:
from santorini.SantoriniGame import SantoriniGame
from santorini.pytorch.NNet import NNetWrapper

game = SantoriniGame(5)
nnet = NNetWrapper(game)
nnet.load_checkpoint(BOOTSTRAP_WORK, BOOTSTRAPPED_FILE)
board = game.getCanonicalForm(game.getInitBoard(), 1)
policy, value = nnet.predict(board)
print("policy shape:", policy.shape)
print("policy sum:", float(policy.sum()))
print("value:", value)
valids = game.getValidMoves(board, 1)
print("legal policy mass:", float((policy * valids).sum()))

## 8. Optional Short Self-Play Resume

After bootstrap succeeds, copy the bootstrapped checkpoint to `best.pth.tar` in a normal checkpoint folder and run a short AlphaZero loop with the translated replay loaded. This is a smoke test, not the long continuation run.

In [ ]:
CHECKPOINT = "/kaggle/working/Santorini-AZ/v2_checkpoints"
os.makedirs(CHECKPOINT, exist_ok=True)
shutil.copy2(os.path.join(BOOTSTRAP_WORK, BOOTSTRAPPED_FILE), os.path.join(CHECKPOINT, "best.pth.tar"))

examples_dst = os.path.join(CHECKPOINT, "latest.examples")
if os.path.lexists(examples_dst):
    os.remove(examples_dst)
try:
    os.symlink(os.path.join(BOOTSTRAP_WORK, BOOTSTRAP_EXAMPLES), examples_dst)
except OSError:
    shutil.copy2(os.path.join(BOOTSTRAP_WORK, BOOTSTRAP_EXAMPLES), examples_dst)

!ls -lh "$CHECKPOINT"

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --load-folder "$CHECKPOINT" \
  --load-model \
  --load-examples \
  --self-play-batch-size 32 \
  --batch-size 128 \
  --arena-batch-size 16 \
  --num-iters 1 \
  --num-eps 8 \
  --num-mcts-sims 16 \
  --arena-compare 8 \
  --history-iters 20 \
  --update-threshold 0.50 \
  --epochs 1 \
  --quiet

## 9. Save Outputs

After the notebook finishes, save or version the notebook output containing `/kaggle/working/Santorini-AZ`. On the next run, attach that saved output or Dataset and point resume paths at its checkpoint folder under `/kaggle/input`.

In [ ]:
!du -sh /kaggle/working/Santorini-AZ || true
!find /kaggle/working/Santorini-AZ -maxdepth 3 -type f -printf "%p %k KB\n" | sort | tail -50